In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

con = duckdb.connect("../data/processed/chess.duckdb")

games = con.sql("""
SELECT *
FROM games_matched
""").df()

tournaments = con.sql("""
SELECT *
FROM tournaments
""").df()

rating_history = con.sql("""
SELECT *
FROM rating_history
""").df()

print(f"Games: {len(games):,}")
print(f"Tournaments: {len(tournaments):,}")
print(f"Rating History: {len(rating_history):,}")

Games: 483,962
Tournaments: 2,098
Rating History: 418,945


In [2]:
top_elo = (
    rating_history
    .groupby("player_name")
    .agg(
        avg_rating=("rating", "mean"),
        games=("rating", "count")
    )
)

top_elo = (
    top_elo
    .query("games >= 20")
    .sort_values(
        "avg_rating",
        ascending=False
    )
    .head(20)
)

top_elo

,avg_rating,games
player_name,,
"Carlsen,M",2837.225806,124
"Nakamura,Hi",2805.588235,136
"Caruana,F",2790.317073,205
"Erigaisi,Arjun",2766.170732,205
"Abdusattorov,Nodirbek",2763.060403,149
"Keymer,Vincent",2760.563218,174
"Firouzja,Alireza",2759.782946,129
"Praggnanandhaa,R",2758.171975,157
"Giri,A",2757.436242,149


In [3]:
fig = px.bar(
    top_elo.reset_index(),
    x="player_name",
    y="avg_rating",
    title="Top Players by Average Elo"
)

fig.show()

In [4]:
most_active = (
    rating_history
    .groupby("player_name")
    .size()
    .reset_index(name="games")
    .sort_values(
        "games",
        ascending=False
    )
    .head(20)
)

most_active

,player_name,games
21738,"Pribelszky,Bence",261
23958,"Sarana,A",227
478,Ajay Santhosh Parvathareddy,218
3247,"Bluebaum,M",215
21671,"Pranav,V",212
6836,"Donchenko,Alexa",211
2818,"Berczes,D",207
7542,"Erigaisi,Arjun",205
4505,"Caruana,F",205
21676,"Pranesh,M",202


In [5]:
fig = px.bar(
    most_active,
    x="player_name",
    y="games",
    title="Most Active Players"
)

fig.show()

In [6]:
def plot_player(player_name):

    df = rating_history[
        rating_history["player_name"]
        == player_name
    ].sort_values("date")

    fig = px.line(
        df,
        x="date",
        y="rating",
        title=f"Rating History - {player_name}"
    )

    fig.show()

In [7]:
plot_player("Caruana,F")

In [8]:
plot_player("Erigaisi,Arjun")

In [9]:
plot_player("Carlsen,M")

In [10]:
strongest = (
    tournaments
    .sort_values(
        "avg_elo",
        ascending=False
    )
    .head(20)
)

strongest[
    [
        "event",
        "year",
        "avg_elo",
        "players_count",
        "games_count"
    ]
]

,event,year,avg_elo,players_count,games_count
1632,TCEC 29 Superfinal 2026,2026.0,3785.0,2.0,100
1631,TCEC 29 Premier 2026,2026.0,3726.4,8.0,224
1630,TCEC 29 Gambit Bonus 2026,2026.0,3645.0,2.0,100
1628,TCEC 28 Premier 2025,2025.0,3640.0,8.0,224
1119,Freestyle Vegas Lower SF,2025.0,2799.5,4.0,6
951,Clutch Chess: Champions,2025.0,2799.0,4.0,36
1116,Freestyle Vegas Int Green,2025.0,2798.0,4.0,6
322,31st Sigeman & Co TB,2026.0,2795.5,2.0,3
1123,Freestyle Vegas White TB,2025.0,2790.5,2.0,2
1124,Freestyle WCh KO 2026,2026.0,2790.5,4.0,16


In [11]:
engine_words = [
    "stockfish",
    "tcec",
    "lczero",
    "komodo",
    "berserk",
    "ethereal"
]

human_tournaments = tournaments[
    ~tournaments["event"]
    .str.lower()
    .str.contains(
        "|".join(engine_words),
        na=False
    )
]

human_tournaments = (
    human_tournaments
    .sort_values(
        "avg_elo",
        ascending=False
    )
)

human_tournaments.head(20)

,event,site,year,start_date,end_date,games_count,avg_elo,draw_rate,decisive_rate,players_count
1119,Freestyle Vegas Lower SF,Las Vegas USA,2025.0,2025-07-19,2025-07-19,6,2799.5,33.33,66.67,4.0
951,Clutch Chess: Champions,Saint Louis USA,2025.0,2025-10-27,2025-10-29,36,2799.0,36.11,63.89,4.0
1116,Freestyle Vegas Int Green,Las Vegas USA,2025.0,2025-07-19,2025-07-19,6,2798.0,16.67,83.33,4.0
322,31st Sigeman & Co TB,Malmo SWE,2026.0,2026-05-07,2026-05-07,3,2795.5,0.00,100.00,2.0
1123,Freestyle Vegas White TB,Las Vegas USA,2025.0,2025-07-16,2025-07-16,2,2790.5,0.00,100.00,2.0
1124,Freestyle WCh KO 2026,Wangels GER,2026.0,2026-02-14,2026-02-15,16,2790.5,56.25,43.75,4.0
1923,World Blitz Final KO 2025,Doha QAT,2025.0,2025-12-30,2025-12-30,11,2788.6,36.36,63.64,4.0
1007,Esports World Cup 2025,Riyadh KSA,2025.0,2025-07-31,2025-08-01,41,2787.5,29.27,70.73,8.0
952,Clutch Chess: The Legends,Saint Louis USA,2025.0,2025-10-08,2025-10-10,12,2777.5,50.00,50.00,2.0
1115,Freestyle Vegas Finals,Las Vegas USA,2025.0,2025-07-20,2025-07-20,8,2776.0,37.50,62.50,8.0


In [12]:
human_tournaments[
    [
        "event",
        "year",
        "draw_rate",
        "avg_elo"
    ]
].sort_values(
    "draw_rate",
    ascending=False
).head(20)

,event,year,draw_rate,avg_elo
1164,Gotham vs. Song Classical,2026.0,100.00,2321.5
1431,Pioneer Youth Elite Cup,2026.0,100.00,2452.6
584,59th Biel GMT Cl Final,2026.0,100.00,2706.2
589,59th Biel INV Cl Final,2026.0,100.00,2566.7
50,13th Sinquefield Cup TB,2026.0,100.00,2757.5
1379,Nakamura-Liang Classical,2026.0,100.00,2762.0
1518,SS4 Springare JGP Match,2026.0,100.00,2505.0
1430,Pioneer Youth Elite Cup,2025.0,90.00,2431.4
1839,Tournament of Peace 2024,2024.0,84.44,2518.7
1383,Navara - Blohberger Match,2026.0,83.33,2593.5


In [13]:
human_tournaments[
    [
        "event",
        "year",
        "decisive_rate",
        "avg_elo"
    ]
].sort_values(
    "decisive_rate",
    ascending=False
).head(20)

,event,year,decisive_rate,avg_elo
1192,Gukesh vs Duda Blitz 2025,2025.0,100.0,2750.5
1378,Nakamura-Liang Blitz,2026.0,100.0,2762.0
1380,Nakamura-Liang Rapid,2026.0,100.0,2762.0
644,76th ch-ARG w TB 2025,2025.0,100.0,2150.5
2085,ch-USA Senior Women Cancelled 2026,2026.0,100.0,2049.5
892,CHN YanChen GM Rapid,2025.0,100.0,2745.0
2,104th ch-IRL TB Final,2025.0,100.0,2185.0
1847,Turkish Cup TB 2025,2025.0,100.0,2253.0
579,58th Iranian Men 1405,2026.0,100.0,2254.6
935,Chess.com Open PlayIn3 KO,2026.0,100.0,2733.5


In [14]:
games["elo_diff"] = (
    games["white_elo"]
    - games["black_elo"]
)

upsets = games[
    (
        (games["elo_diff"] >= 100)
        &
        (games["result"] == "0-1")
    )
    |
    (
        (games["elo_diff"] <= -100)
        &
        (games["result"] == "1-0")
    )
].copy()

upsets["upset_size"] = (
    upsets["elo_diff"]
    .abs()
)

upsets = upsets.sort_values(
    "upset_size",
    ascending=False
)

upsets[
    [
        "date",
        "white",
        "black",
        "white_elo",
        "black_elo",
        "result",
        "upset_size"
    ]
].head(20)

,date,white,black,white_elo,black_elo,result,upset_size
336516,2026-06-17,"Yip,Carissa","Leung,Kwan Yat Calix",2458.0,1536.0,0-1,922.0
302871,2026-05-09,"Lashkin,Jegor","Caraiman,Cezar-Cristian",2475.0,1557.0,0-1,918.0
174932,2025-12-29,"AlJamaat,Khaled","Bjerre,Jonas Buhl",1719.0,2623.0,1-0,904.0
336570,2026-06-17,"Da Silva,Marco","Tin,Jingyao",1687.0,2589.0,1-0,902.0
336571,2026-06-17,"Injac,Teodora","Anandpara,Jayendra Saloni",2430.0,1555.0,0-1,875.0
121070,2025-10-24,"Simeonov,Velin","Trifunov,D",1551.0,2403.0,1-0,852.0
56561,2025-08-12,"Alazmi,Amani","Tregubov,P",1750.0,2568.0,1-0,818.0
418182,2026-08-20,Bui Quang Huy,"Martinez Alcantara,Jose Eduardo",1850.0,2644.0,1-0,794.0
338334,2026-06-20,"Tin,Jingyao","Fang,Jack Junrui",2589.0,1809.0,0-1,780.0
133767,2025-11-14,"Terekhov,A",Nerkar Anand,2261.0,1493.0,0-1,768.0
